In [2]:
import numpy as np
import pandas as pd
import time
from tqdm import tqdm 
import os

In [3]:
from coral_ren import *

In [4]:
root = '/home/cz/'

In [5]:
path = os.path.join(root, 'mds3/REN/datasets_zeroshot/prepared')

In [6]:
path2save = os.path.join(root, 'mds3/REN/few-shot/OUTPUTS/Zeroshot/Coral')
os.makedirs(path2save,exist_ok=True)

In [7]:
texts = np.load(path+'/texts_test_v1.npy',allow_pickle=True) 

In [12]:
np.load(os.path.join(path2save, 'predicted_entities-2.npy'),allow_pickle=True)

array([['Brasil', 'LOC'],
       ['Washington', 'LOC'],
       ['EUA', 'LOC'],
       ['Chile', 'LOC'],
       ['Argentina', 'LOC'],
       ['Otaviano Canuto', 'PESSOA'],
       ['Joe Biden', 'PESSOA'],
       ['Canuto', 'PESSOA'],
       ['Banco Mundial', 'ORG'],
       ['Fundo Monetário Internacional', 'ORG'],
       ['FMI', 'ORG'],
       ['Center for Macroeconomics and Development', 'ORG'],
       ['Ministério da Fazenda', 'ORG'],
       ['Federal Reserve', 'ORG'],
       ['Fed', 'ORG'],
       ['Banco Central', 'ORG'],
       ['BC', 'ORG'],
       ['Copom', 'ORG'],
       ['Tesouro', 'ORG']], dtype='<U41')

In [13]:
path2save = os.path.join(root, 'mds3/REN/few-shot/OUTPUTS/Zeroshot/Coral/Tips')
np.load(os.path.join(path2save, 'predicted_entities-2.npy'),allow_pickle=True)

array([['Brasil', 'LOC'],
       ['Washington', 'LOC'],
       ['EUA', 'LOC'],
       ['Chile', 'LOC'],
       ['Argentina', 'LOC'],
       ['Otaviano Canuto', 'PESSOA'],
       ['Joe Biden', 'PESSOA'],
       ['Canuto', 'PESSOA'],
       ['Banco Mundial', 'ORG'],
       ['Fundo Monetário Internacional', 'ORG'],
       ['FMI', 'ORG'],
       ['Center for Macroeconomics and Development', 'ORG'],
       ['Ministério da Fazenda', 'ORG'],
       ['Federal Reserve', 'ORG'],
       ['Fed', 'ORG'],
       ['Banco Central', 'ORG'],
       ['BC', 'ORG'],
       ['Copom', 'ORG'],
       ['Tesouro', 'ORG']], dtype='<U41')

In [8]:
targets = texts = np.load(path+'/labels_test_v1.npy',allow_pickle=True).item()

In [ ]:
checkpoint_file = os.path.join(path2save, 'checkpoint.txt')

# Carregar o índice de onde continuar
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        start_index = int(f.read().strip())+1
else:
    start_index = 0

# Loop através dos textos, começando do índice salvo
for i in tqdm(range(start_index, len(texts))):

    text = texts[i]
    labels = list(targets[i].keys())

    entities = extract_entities(text)
    print(entities)

     # Loop para tentar converter até ter sucesso
    try:
        predicted_entities = converting(entities)

        # Salvar as entidades preditas
        np.save(os.path.join(path2save, f'predicted_entities-{i}.npy'), predicted_entities)

        # Salvar o índice atual no checkpoint
        with open(checkpoint_file, 'w') as f:
            f.write(str(i))

    except Exception as e:
        print(f"Erro ao converter entidades: {e}. Tentando novamente...")
        time.sleep(5)  # Espera um pouco antes de tentar novamente
        continue
    time.sleep(45)  # Espera antes de passar para o próximo texto


In [66]:
for i in tqdm(range(11,12)):

    text = texts[i]
    labels = list(targets[i].keys())

    entities = extract_entities(text)
    print(entities)

100%|██████████| 1/1 [00:07<00:00,  7.66s/it]

As entidades nomeadas na frase são:

- Gaza (LOC)
- Israel (LOC)
- Jerusalém (LOC)
- Tel-Aviv (LOC)
- Binyamin Netanyahu (PESSOA)
- Yair Lapid (PESSOA)
- Netanyahu (PESSOA)
- Lapid (PESSOA)
- Reuven Hazan (PESSOA)
- Mansour Abbas (PESSOA)
- Abbas (PESSOA)
- Mitchell Barak (PESSOA)
- Benny Gantz (PESSOA)
- Dahlia Scheindlin (PESSOA)
- Facebook (ORG)
- Universidade Hebraica de Jerusalém (ORG)

As classes para cada entidade são:

- LOC (Localização)
- PESSOA
- ORG (Organização)


In [67]:
fixed = entities.replace('As classes para cada entidade são:','')

In [72]:
fixed = fixed.replace('As entidades nomeadas na frase são:','').replace('- LOC (Localização)\n- PESSOA\n- ORG (Organização)','')

In [61]:
def identificar_entidades_com_parenteses(texto):
    entidades_label = []
    
    # Dividir o texto em linhas
    linhas = texto.split("\n")
    
    for linha in linhas:
        linha = linha.strip()
        if linha:  # Ignora linhas vazias
            # Dividir a linha em entidade e label usando os parênteses
            entidade, label = linha.rsplit(' (', 1)
            label = label.replace(")", "")  # Remover o parêntese final do label
            entidades_label.append((label.strip(),entidade.replace('-','').strip() ))
    
    return entidades_label

In [78]:
(set([('a','b'),('c','e')]) & set([('b','a'),('c','e')]))

{('c', 'e')}

In [79]:
# Definindo os conjuntos com tuplas ordenadas
 set(tuple(sorted(pair)) for pair in [('a', 'b'), ('c', 'e')])
 set(tuple(sorted(pair)) for pair in [('b', 'a'), ('c', 'e')])

# Fazendo a interseção
interseccao = set1 & set2

print(interseccao)  # Saída: {('a', 'b'), ('c', 'e')}


{('c', 'e'), ('a', 'b')}


In [82]:
set1,set2

({('a', 'b'), ('c', 'e')}, {('a', 'b'), ('c', 'e')})

In [73]:
right = identificar_entidades_com_parenteses(texto= fixed)

In [74]:
right

[('LOC', 'Gaza'),
 ('LOC', 'Israel'),
 ('LOC', 'Jerusalém'),
 ('LOC', 'TelAviv'),
 ('PESSOA', 'Binyamin Netanyahu'),
 ('PESSOA', 'Yair Lapid'),
 ('PESSOA', 'Netanyahu'),
 ('PESSOA', 'Lapid'),
 ('PESSOA', 'Reuven Hazan'),
 ('PESSOA', 'Mansour Abbas'),
 ('PESSOA', 'Abbas'),
 ('PESSOA', 'Mitchell Barak'),
 ('PESSOA', 'Benny Gantz'),
 ('PESSOA', 'Dahlia Scheindlin'),
 ('ORG', 'Facebook'),
 ('ORG', 'Universidade Hebraica de Jerusalém')]

In [75]:
pth_root = '/home/cz'
path_ren  = 'mds3/REN/few-shot/OUTPUTS/Zeroshot'
full_path_ren = os.path.join(pth_root, path_ren)


In [76]:
np.save(os.path.join(full_path_ren,'Coral',f'predicted_entities-{11}.npy'),right)